# Otimização Multi-objetivo

- Algoritmos offline (NSGA-II, DR-NSGA-II, Prob-MOEA/D, UA-SA-NSGA2): usam landscape do surrogate (n_var=2, n_obj=2)
- Algoritmos online (K-RVEA, KTA2, ParEGO): usam `NoisyProblem` wrapper para buscar sobre fitness ruidosas
- Todos os resultados são re-avaliados com a função verdadeira para comparação justa

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning

from src.experiment import (
    experiment_sa_moea,
    ALGORITHM_DISPATCH,
    ALL_PROBLEMS,
    BASE_CONFIG,
    NOISE_CONFIG,
    _instantiate_problem,
)
from src.processing import NoisyProblem
from src.visualization import (
    display_pareto_fronts_catalog,
    display_decision_space,
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['lines.linewidth'] = 0.5
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# ═══════════════════════════════════════════════════════════════════
#  Random seed and problem catalog
# ═══════════════════════════════════════════════════════════════════
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 26 problemas (catalogo completo de src/problems.py)
PROBLEM_NAMES = ALL_PROBLEMS

# ═══════════════════════════════════════════════════════════════════
#  Visualization & algorithm selection
# ═══════════════════════════════════════════════════════════════════
# dict_display controla quais algoritmos sao desenhados nos plots,
# e com qual cor.  Comente/descomente para mostrar/ocultar.
dict_display = {
    'NSGA2_surrogate': 'yellow',
    'DR-NSGA-II':      'lime',
    'Prob-MOEA/D':     'green',
    'K-RVEA':          'cyan',
    'ParEGO':          'orange',
    'KTA2':            'fuchsia',
}

# dict_algorithms controla quais algoritmos sao executados.
# Mantenha True para incluir no run, False para pular.
dict_algorithms = {algo: True for algo in ALGORITHM_DISPATCH}

# ═══════════════════════════════════════════════════════════════════
#  Modo Rapido (quick-test flag)
# ═══════════════════════════════════════════════════════════════════
MODO_RAPIDO = False #True
fator_modo_rapido = 1 #0.1

# Build local config for this notebook run (does NOT mutate BASE_CONFIG global)
nb_config = BASE_CONFIG.copy()
if MODO_RAPIDO:
    for k in ('n_generations', 'krvea_feval', 'kta2_feval', 'parego_feval'):
        nb_config[k] = max(1, int(nb_config[k] * fator_modo_rapido))
    nb_config['FEmax'] = nb_config.get('krvea_feval', 300)

# Re-export noise_config for backwards compat references
noise_config = NOISE_CONFIG

# ═══════════════════════════════════════════════════════════════════
#  Metrics (HV, IGD, Gamma, Delta)
# ═══════════════════════════════════════════════════════════════════

def compute_hypervolume(F, ref_point):
    """Hypervolume indicator (S-metric). Used in: K-RVEA, DR-NSGA-II, ParEGO, Prob-MOEA/D."""
    from pymoo.indicators.hv import HV
    ind = HV(ref_point=ref_point)
    return ind(F)


def compute_igd(F_approx, F_true):
    """Inverted Generational Distance.  Used in: K-RVEA, KTA2."""
    from scipy.spatial.distance import cdist
    D = cdist(F_true, F_approx)
    return np.mean(np.min(D, axis=1))


def compute_gamma(F_approx, F_true):
    """Gamma convergence metric (Deb et al., 2002 -- NSGA-II paper)."""
    from scipy.spatial.distance import cdist
    D = cdist(F_approx, F_true)
    return np.mean(np.min(D, axis=1))


def compute_delta(F_approx, F_true):
    """Delta diversity metric (Deb et al., 2002 -- NSGA-II paper)."""
    if len(F_approx) < 2:
        return float('inf')
    from scipy.spatial.distance import cdist
    n_obj = F_approx.shape[1]
    idx_sort = np.lexsort(F_approx[:, ::-1].T)
    F_sorted = F_approx[idx_sort]
    d = np.array([np.linalg.norm(F_sorted[i+1] - F_sorted[i])
                  for i in range(len(F_sorted) - 1)])
    if len(d) == 0:
        return float('inf')
    d_bar = d.mean()
    D_to_true = cdist(F_true, F_approx)
    extremes_true_idx = []
    for j in range(n_obj):
        extremes_true_idx.append(np.argmin(F_true[:, j]))
    d_f = np.min(D_to_true[extremes_true_idx[0]])
    d_l = np.min(D_to_true[extremes_true_idx[-1]])
    denom = d_f + d_l + (len(d)) * d_bar
    if denom == 0:
        return 0.0
    return (d_f + d_l + np.sum(np.abs(d - d_bar))) / denom


def evaluate_metrics(F_approx, problem, n_pf_points=500):
    """Compute all 4 metrics for an approximation set."""
    _, F_true = problem.true_pareto_front(n_pf_points)
    ref_point = np.max(F_true, axis=0) * 1.1 + 0.01
    metrics = {}
    try:
        metrics['HV'] = float(compute_hypervolume(F_approx, ref_point))
    except Exception:
        metrics['HV'] = None
    metrics['IGD'] = float(compute_igd(F_approx, F_true))
    metrics['Gamma'] = float(compute_gamma(F_approx, F_true))
    metrics['Delta'] = float(compute_delta(F_approx, F_true))
    return metrics


# Global dict to accumulate results across all problems
ALL_METRICS = {}


# ═══════════════════════════════════════════════════════════════════
#  Visualisation
# ═══════════════════════════════════════════════════════════════════

def visualize_optimization_results(name, problem, df_landscape_data, returns):
    """Plot Pareto fronts and decision-space for all algorithms in ``returns``.

    ``returns`` is a dict {algo_name: df_pareto} where df_pareto has columns
    x_1..x_n and f1..fm (last-generation slice of the long DataFrame).
    Only algorithms also present in ``dict_display`` are shown.
    """
    n_obj = problem.n_obj
    n_var = problem.n_var
    x_cols = [f'x_{i+1}' for i in range(n_var)]
    f_cols = [f'f{j+1}' for j in range(n_obj)]

    X_true, _ = problem.true_pareto_front()
    F_landscape = df_landscape_data[f_cols].values
    X_landscape = df_landscape_data[x_cols].values

    algo_names, algo_F, algo_X, algo_colors = [], [], [], []
    for algo_name, color in dict_display.items():
        if algo_name not in returns:
            continue
        df_res = returns[algo_name]
        avail_f = [c for c in f_cols if c in df_res.columns]
        avail_x = [c for c in x_cols if c in df_res.columns]
        if len(avail_f) != n_obj:
            print(f'  Warn: {algo_name}: colunas de objetivo incompletas, pulando visualizacao')
            continue
        algo_names.append(algo_name)
        algo_F.append(df_res[avail_f].values)
        algo_X.append(df_res[avail_x].values)
        algo_colors.append(color)

    if not algo_names:
        print(f'  Sem resultados para visualizar em {name}')
        return

    print('\n' + '=' * 80)
    print(f'{name} -- ESPACO DE OBJETIVOS')
    print('=' * 80)

    display_pareto_fronts_catalog(
        problem, F_landscape,
        pareto_fronts_list=algo_F,
        front_names=algo_names,
        front_colors=algo_colors,
        sample_size=100_000,
        title=f'{name} -- Espaco de Objetivos',
    )

    print('\n' + '=' * 80)
    print(f'{name} -- ESPACO DE DECISOES')
    print('=' * 80)

    display_decision_space(
        problem, X_landscape, F_landscape,
        X_true=X_true,
        X_emp_list=algo_X,
        emp_names=algo_names,
        emp_colors=algo_colors,
        cmap='plasma',
        title=name,
    )


/Users/gmello/Documents/python_venvs/mestrado_estatistica/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Execução de Experimentos

Executa `experiment_sa_moea` (de `src/experiment.py`) para cada par
(algoritmo, problema) com `seed=42`. O resultado é uma tabela única
em formato long com a população completa de cada geração, pronta para
visualização e cálculo de métricas (sobre a última geração) e para
alimentar o dashboard Streamlit (todas as gerações).


In [ ]:
SEED = 42
OUT_DIR = Path('data/experiments')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'notebook_seed42.parquet'

# Toggle: True = pula problemas/algos cujo run ja existe individualmente em data/experiments/single/
USAR_CACHE_SINGLE = True

ALL_DFS = []
for problem in PROBLEM_NAMES:
    print(f'\n{"#" * 80}\n# {problem}\n{"#" * 80}')
    for algo, enabled in dict_algorithms.items():
        if not enabled:
            continue
        cache_path = OUT_DIR / 'single' / f"{algo.replace('/', '_').replace(' ', '_').replace('-', '_').lower()}_{problem}_seed{SEED}.parquet"
        if USAR_CACHE_SINGLE and cache_path.exists():
            df = pd.read_parquet(cache_path)
            print(f'  [cache] {algo:18s} -> {df.shape}')
        else:
            print(f'  [run]   {algo:18s} ...', end=' ', flush=True)
            df = experiment_sa_moea(algo, problem, SEED, config=nb_config)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_parquet(cache_path)
            print(f'shape={df.shape}')
        ALL_DFS.append(df)

df_all = pd.concat(ALL_DFS, ignore_index=True)
df_all.to_parquet(OUT_PATH)
print(f'\nSaved consolidated: {OUT_PATH}  shape={df_all.shape}')
df_all.head()



################################################################################
# MMF1
################################################################################
  [cache] NSGA2_surrogate    -> (1100, 9)
  [run]   DR-NSGA-II         ... 

/Users/gmello/Documents/python_venvs/mestrado_estatistica/lib/python3.12/site-packages/scipy/stats/_qmc.py:958: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)
100%|██████████| 100/100 [00:06<00:00, 16.58it/s]

shape=(10100, 9)
  [run]   Prob-MOEA/D        ... 


/Users/gmello/Documents/python_venvs/mestrado_estatistica/lib/python3.12/site-packages/scipy/stats/_qmc.py:958: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)
100%|██████████| 100/100 [02:30<00:00,  1.50s/it]

shape=(10100, 9)
  [cache] K-RVEA             -> (102, 9)
  [run]   ParEGO             ... 

...

/Users/gmello/Documents/python_venvs/mestrado_estatistica/lib/python3.12/site-packages/scipy/stats/_qmc.py:958: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)
100%|██████████| 100/100 [00:08<00:00, 12.29it/s]

shape=(10100, 11)
  [run]   Prob-MOEA/D        ... 


/Users/gmello/Documents/python_venvs/mestrado_estatistica/lib/python3.12/site-packages/scipy/stats/_qmc.py:958: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)
100%|██████████| 100/100 [2:22:20<00:00, 85.40s/it] 


shape=(510050, 11)
  [run]   K-RVEA             ... 

K-RVEA Online (FE):  23%|██▎       | 68/300 [6:05:24<37:51:52, 587.55s/it]

## Visualização e Métricas

Para cada problema:
- Extrai a **última geração** do long-DataFrame (front final por algoritmo)
- Plota o espaço de objetivos e o espaço de decisões
- Calcula as 4 métricas (HV, IGD, Γ, Δ) acumulando em `ALL_METRICS`


In [ ]:
def _last_gen_slice(df_long, algo, problem):
    sub = df_long[(df_long.algorithm == algo) & (df_long.problem == problem)]
    if sub.empty:
        return sub
    return sub[sub.generation == sub.generation.max()].copy()


for problem in PROBLEM_NAMES:
    p_obj = _instantiate_problem(problem)
    base = Path('data/dataframes') / problem
    landscape_path = base / f'df_{problem}_landscape_sobol.parquet'
    if not landscape_path.exists():
        print(f'  Skipping {problem}: landscape parquet ausente em {landscape_path}')
        continue
    df_landscape_data = pd.read_parquet(landscape_path)

    sub_problem = df_all[df_all.problem == problem]
    available_algos = sub_problem.algorithm.unique()

    returns = {}
    for algo in available_algos:
        df_last = _last_gen_slice(df_all, algo, problem)
        if df_last.empty:
            continue
        returns[algo] = df_last

    print(f'\n{"#" * 80}\n# {problem}\n{"#" * 80}')
    visualize_optimization_results(problem, p_obj, df_landscape_data, returns)

    # Metrics
    print('\n' + '-' * 60)
    print(f'{problem} -- METRICAS')
    print('-' * 60)
    f_cols = [f'f{j+1}' for j in range(p_obj.n_obj)]
    problem_metrics = {}
    for algo, df_res in returns.items():
        avail_f = [c for c in f_cols if c in df_res.columns]
        if len(avail_f) != p_obj.n_obj:
            problem_metrics[algo] = None
            continue
        F_algo = df_res[avail_f].values.astype(float)
        m = evaluate_metrics(F_algo, p_obj)
        problem_metrics[algo] = m
        print(f'  {algo:20s}  HV={m["HV"]:.4f}  IGD={m["IGD"]:.4f}  '
              f'Gamma={m["Gamma"]:.4f}  Delta={m["Delta"]:.4f}')
    ALL_METRICS[problem] = problem_metrics


## Métricas de Avaliação

As 4 métricas abaixo foram selecionadas com base na frequência de uso nos artigos dos algoritmos implementados e na cobertura de diferentes aspectos da qualidade das soluções (convergência e diversidade).

### 1. Hypervolume (HV) — Convergência + Diversidade | ↑ Maior = melhor
**Aparece em:** K-RVEA, DR-NSGA-II, ParEGO, Prob-MOEA/D (4 artigos)

Volume do espaço de objetivos **dominado** pelo conjunto de soluções A, limitado por um ponto de referência r (pior que todos os pontos). Um conjunto com melhor convergência E melhor espalhamento terá maior HV.

**Cálculo:** Para cada par de pontos consecutivos (ordenados por f₁), calcula-se a área/volume do retângulo/hipercubo dominado. HV = soma dessas áreas.

**Exemplo 2D:** A = {(0.2, 0.8), (0.5, 0.3), (0.9, 0.1)}, r = (1.1, 1.1)
- Retângulo 1: (1.1−0.2) × (1.1−0.8) = 0.27
- Retângulo 2: (1.1−0.5) × (0.8−0.3) = 0.30
- Retângulo 3: (1.1−0.9) × (0.3−0.1) = 0.04
- **HV = 0.61**

### 2. IGD (Inverted Generational Distance) — Convergência + Diversidade | ↓ Menor = melhor
**Aparece em:** K-RVEA, KTA2 (2 artigos)

Para cada ponto `p` do PF verdadeiro P*, encontra a distância ao ponto mais próximo no conjunto A.

**Cálculo:** IGD = (1/|P*|) × Σ_{p ∈ P*} min_{a ∈ A} ‖p − a‖

**Exemplo:** P* = {(0,1), (0.5,0.5), (1,0)}, A = {(0.1,0.9), (0.6,0.4)}
- d(p₁) = min(0.141, 0.781) = 0.141
- d(p₂) = min(0.566, 0.141) = 0.141
- d(p₃) = min(1.273, 0.566) = 0.566
- **IGD = 0.283**

Se A não cobre uma região do PF verdadeiro, os pontos nessa região terão distâncias altas → IGD sobe.

### 3. Gamma (Γ) — Convergência pura | ↓ Menor = melhor
**Aparece em:** NSGA-II (Deb et al., 2002, Sec. IV-B)

Inverso do IGD: para cada ponto `a` do conjunto encontrado A, distância ao PF verdadeiro mais próximo.

**Cálculo:** Γ = (1/|A|) × Σ_{a ∈ A} min_{p ∈ P*} ‖a − p‖

Se todas as soluções estão exatamente no PF, Γ = 0.

### 4. Delta (Δ) — Diversidade / Uniformidade | ↓ Menor = melhor
**Aparece em:** NSGA-II (Deb et al., 2002, Sec. IV-B)

Mede a uniformidade da distribuição de soluções ao longo do front.

**Cálculo:** Δ = (d_f + d_l + Σ|d_i − d̄|) / (d_f + d_l + (N−1)×d̄)

Onde:
- d_i = distância entre soluções consecutivas (ordenadas por f₁)
- d̄ = média das d_i
- d_f = distância do extremo do PF verdadeiro ao primeiro ponto de A
- d_l = distância do extremo do PF verdadeiro ao último ponto de A

Se todas as d_i = d̄ (perfeitamente uniforme) e d_f = d_l = 0 (cobre os extremos): **Δ = 0**.

## Sumário de Resultados

In [ ]:
# Construir tabela de métricas: linhas = problemas, colunas = algoritmos
import pandas as pd

metric_names = ['HV', 'IGD', 'Gamma', 'Delta']
all_algos = sorted(set(algo for pm in ALL_METRICS.values() for algo in pm))

rows = []
for prob_name, algo_dict in ALL_METRICS.items():
    for algo in all_algos:
        if algo in algo_dict and algo_dict[algo] is not None:
            m = algo_dict[algo]
            rows.append({
                'Problema': prob_name,
                'Algoritmo': algo,
                'HV ↑': f'{m["HV"]:.4f}' if m['HV'] is not None else '-',
                'IGD ↓': f'{m["IGD"]:.4f}',
                'Γ (conv.) ↓': f'{m["Gamma"]:.4f}',
                'Δ (div.) ↓': f'{m["Delta"]:.4f}',
            })
        else:
            rows.append({
                'Problema': prob_name,
                'Algoritmo': algo,
                'HV ↑': f'não executado',
                'IGD ↓': f'não executado',
                'Γ (conv.) ↓': f'não executado',
                'Δ (div.) ↓': f'não executado',
            })

df_summary = pd.DataFrame(rows)
df_pivot = df_summary.pivot(index='Problema', columns='Algoritmo',
                            values=['HV ↑', 'IGD ↓', 'Γ (conv.) ↓', 'Δ (div.) ↓'])

# Reorder problems
prob_order = [p for p in PROBLEM_NAMES if p in df_pivot.index]
df_pivot = df_pivot.loc[prob_order]

print('═' * 100)
print('SUMÁRIO DE MÉTRICAS — TODOS OS PROBLEMAS × ALGORITMOS')
print('═' * 100)
print('HV ↑ = Hypervolume (maior melhor)')
print('IGD ↓ = Inverted Generational Distance (menor melhor)')
print('Γ ↓ = Gamma — convergência (menor melhor)')
print('Δ ↓ = Delta — diversidade (menor melhor)')
print()

for metric in ['HV ↑', 'IGD ↓', 'Γ (conv.) ↓', 'Δ (div.) ↓']:
    print(f'\n{"=" * 80}')
    print(f'  {metric}')
    print(f'{"=" * 80}')
    sub = df_pivot[metric]
    display(sub)
